# Catchment analysis for quantifying how NbS reduce river flood risk

### Step 0: Import packages to work with, set up folder pathways and project to Jamaica's grid coordinates

In [ ]:
import geopandas as gpd
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Inputs")
output_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Processed_data")
jamaica_metric_grid_crs = "EPSG:3448"

### Step 1: Read in hydrobasins file, then read in Jamaica boundary and clip hydrobasins to Jamaica

In [ ]:
original_hydrobasins_path = base_path / "Hydrobasins_12/hybas_na_lev12_v1c.shp" 
original_hydrobasins = gpd.read_file(original_hydrobasins_path)

In [ ]:
#print(f"Number of unique HYBAS_ID values: {unique_hybas_ids_original}")
unique_hybas_ids_unclipped = original_hydrobasins['HYBAS_ID'].nunique()
print(f"Number of unique HYBAS_ID values: {unique_hybas_ids_unclipped}")

In [ ]:
jamaica_boundary_path = base_path / "Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(jamaica_boundary.crs)

In [ ]:
# Reproject HydroBASINS to match Jamaica's CRS if necessary
reprojected_hydrobasins = original_hydrobasins.to_crs(jamaica_boundary.crs)
print(original_hydrobasins.crs)
print(reprojected_hydrobasins.crs)

In [ ]:
hydrobasins_clipped = gpd.clip(reprojected_hydrobasins, jamaica_boundary)
hydrobasins_clipped.plot()

In [ ]:
# Check that the hydrobasins and Jamaica boundary are on the right coordinate system and that they are bounded the same (Jamaica bounds)
print("Hydrobasins_clipped CRS:", hydrobasins_clipped.crs)
print("Jamaica Boundary CRS:", jamaica_boundary.crs)

print("Hydrobasins_clipped bounds:", hydrobasins_clipped.total_bounds)
print("Jamaica boundary bounds:", jamaica_boundary.total_bounds)

In [ ]:
output_shapefile = output_path / "HydroBASINS_Level12_Clipped_Jamaica.shp"
hydrobasins_clipped.to_file(output_shapefile)

In [ ]:
# Read the saved shapefile into a GeoDataFrame
hydrobasins_clipped_saved = gpd.read_file(output_shapefile)

In [ ]:
# Count the unique values in the 'HYBAS_ID' column to check how many rows there should be
unique_hybas_ids_original = hydrobasins_clipped_saved['HYBAS_ID'].nunique()

# Print the result
print(f"Number of unique HYBAS_ID values: {unique_hybas_ids_original}")

In [ ]:
display(hydrobasins_clipped_saved.head())

In [ ]:
print("hydrobasins_clipped_saved CRS:", hydrobasins_clipped.crs)
print("hydrobasins_clipped_saved bounds:", hydrobasins_clipped.total_bounds)

### Step 2: Read in land use for Jamaica and intersect it with the clipped and re-projected hydrobasins file

In [ ]:
land_use = gpd.read_file(base_path / "2013_landuse_LandCover.shp")
print(land_use.crs)
land_use.plot()

In [ ]:
land_use_hydrobasins_intersection = gpd.overlay(land_use, hydrobasins_clipped_saved, how='intersection')

In [ ]:
# Define the output shapefile path for the intersected data
output_intersection_shapefile = output_path / "land_use_hydrobasins_intersection.shp"

In [ ]:
# Save the intersected GeoDataFrame to a shapefile
land_use_hydrobasins_intersection.to_file(output_intersection_shapefile)

In [ ]:
# Now, read it back in to confirm it has been saved correctly
land_use_hydrobasins_intersection_saved = gpd.read_file(output_intersection_shapefile)

In [ ]:
# Print the first few rows to confirm the data is loaded
display(land_use_hydrobasins_intersection_saved.head())

In [ ]:
# Count the unique values in the 'HYBAS_ID' column to check how many rows I have
unique_hybas_ids = land_use_hydrobasins_intersection_saved['HYBAS_ID'].nunique()

# Print the result
print(f"Number of unique HYBAS_ID values: {unique_hybas_ids}")

### Step 3: Calculate total area of each catchment and the percentage catchment coverage of each land use type

#### 3.1 Calculate total area of each catchment

In [ ]:
# Calculate the area for each feature in the GeoDataFrame (in square units, based on CRS). 
# The units of the area will depend on the CRS of the data (EPSG: 3448, which uses meters, so the area is in square meters).
land_use_hydrobasins_intersection_saved['area'] = land_use_hydrobasins_intersection_saved.geometry.area

# Total area of each catchmemt - group by 'HYBAS_ID' and sum the 'area' for each catchment
total_area_by_catchment = land_use_hydrobasins_intersection_saved.groupby('HYBAS_ID')['area'].sum().reset_index()
total_catchment_area  = total_area_by_catchment.rename(columns={'area': 'total_catchment_area'})

display(total_catchment_area.head())

#### 3.2 Calculate area of each land use within each catchment

In [ ]:
# Area of land use within each catchmemt - group by 'HYBAS_ID' and 'classify' and sum the 'area' for each catchment
total_landuse_area_by_catchment = land_use_hydrobasins_intersection_saved.groupby(['HYBAS_ID', 'Classify'])['area'].sum().reset_index()

display(total_landuse_area_by_catchment.head())

# Define the output file path
output_landuse_areas_path = output_path / "total_landuse_area_by_catchment.csv"

# Export the DataFrame to a CSV file
total_landuse_area_by_catchment.to_csv(output_landuse_areas_path, index=False)

print(f"Data successfully exported to {output_landuse_areas_path}")

#### 3.3 Calculate percentage catchment covered by each land use type

In [ ]:
# Merge total area of catchments with the land use area to calculate percentage
land_use_with_total_area = total_landuse_area_by_catchment.merge(total_catchment_area, on='HYBAS_ID')

# Calculate the percentage of each land use area relative to the total catchment area
land_use_with_total_area['Percentage of Catchment'] = (land_use_with_total_area['area'] / land_use_with_total_area['total_catchment_area']) * 100

display(land_use_with_total_area.head())

# Define the output file path
output_percentage_landcover_csv_path = output_path / "land_use_percentage_by_catchment.csv"

# Export the DataFrame to a CSV file
land_use_with_total_area.to_csv(output_percentage_landcover_csv_path, index=False)

print(f"Data successfully exported to {output_percentage_landcover_csv_path}")

### Step 4: determine current forest coverage within each catchment and future afforestable area

#### 4.1 Determining current forest, non-afforestable and future afforestable land use categories

In [ ]:
# # Define fractional distribution for land use types including mixed land use

# current_forested_lands = {
#     'Open dry forest - Short',
#     'Open dry forest - Tall (Woodland/Savanna)',
#     'Disturbed broadleaved forest (Secondary Forest)',
#     'Closed broadleaved forest (Primary Forest)',
#     'Secondary Forest',
# }

# non_afforestable = {
#     'Bare Rock',
#     'Herbaceous Wetland',
#     'Mangrove Forest',
#     'Water Body',
#     'Buildings and other infrastructures',
# }

# future_afforestable_non_agricultural_lands = {
#     'Fields: Bare Land',
#     'Quarry',
#     'Bauxite Extraction',
#     'Bamboo',
# }

# future_afforestable_agricultural_lands = {
#     'Fields: Herbaceous crops, fallow, cultivated vegetables',
#     'Fields: Pasture,Human disturbed, grassland',
# }

# mixed_land_use_fractions = {
#     'Fields and Secondary Forest': {'future_afforestable_agricultural_lands': 0.75, 'current_forested_lands': 0.25},
#     'Bamboo and Fields': {'future_afforestable_non_agricultural_lands': 0.75, 'future_afforestable_agricultural_lands': 0.25},
#     'Fields  and Bamboo': {'future_afforestable_agricultural_lands': 0.75, 'future_afforestable_non_agricultural_lands': 0.25},
#     'Bamboo and Secondary Forest': {'future_afforestable_non_agricultural_lands': 0.75, 'current_forested_lands': 0.25},
#     'Fields or Secondary Forest/Pine Plantation': {'future_afforestable_agricultural_lands': 0.75, 'current_forested_lands': 0.25},
# }

# tree_plantations = {
#     'Plantation: Tree crops, shrub crops, sugar cane, banana',
#     'Hardwood Plantation: Euculytus',
#     'Hardwood Plantation: Mixed' 'Swamp Forest',
#     'Hardwood Plantation: Mahoe',
#     'Hardwood Plantation: Mahogany', 
# }

# display(current_forested_lands)

In [ ]:
# Define your desired category lists assuming purely flood reduction (bamboo, plantations)
forest_flood_equivalent_classes = {
    'Open dry forest - Short',
    'Open dry forest - Tall (Woodland/Savanna)',
    'Disturbed broadleaved forest (Secondary Forest)',
    'Closed broadleaved forest (Primary Forest)',
    'Secondary Forest',
    'Fields and Secondary Forest',       # 25% Forest, 75% Ag
    'Bamboo and Secondary Forest',        # 25% Forest, 75% Ag (if you want some ag fraction)
    'Bamboo',
    'Plantation: Tree crops, shrub crops, sugar cane, banana',
    'Hardwood Plantation: Euculytus',
    'Hardwood Plantation: Mixed',
    'Hardwood Plantation: Mahoe',
    'Hardwood Plantation: Mahogany', 
}

afforestable_classes_including_agricultural = {
    'Fields: Herbaceous crops, fallow, cultivated vegetables',
    'Fields: Pasture,Human disturbed, grassland',
    'Fields and Secondary Forest',        # 75% Ag, 25% Forest
    'Bamboo and Fields',                  # 75% Ag, 25% Bamboo (if you consider bamboo here)
    'Fields  and Bamboo',                  # 75% Ag, 25% Bamboo
    'Fields or Secondary Forest/Pine Plantation',  # 75% Ag, 25% Secondary Forest/Pine
    'Fields: Bare Land',
    'Quarry',
    'Bauxite Extraction'
}

# Update the mixed land use fractions using the new keys.
mixed_land_use_fractions = {
    'Fields and Secondary Forest': {
         'forest_flood_equivalent_classes': 0.25,
         'afforestable_including_agriculture': 0.75
    },
    'Bamboo and Secondary Forest': {
         'forest_flood_equivalent_classes': 1
    },
    # If these classes are purely afforestable, you can allocate 100% to that category:
    'Bamboo and Fields': {
         'forest_flood_equivalent_classes': 0.75,
         'afforestable_including_agriculture': 0.25
    },
    'Fields  and Bamboo': {
         'forest_flood_equivalent_classes': 0.25,
         'afforestable_including_agriculture': 0.75
    },
    'Fields or Secondary Forest/Pine Plantation': {
         'forest_flood_equivalent_classes': 0.25,
         'afforestable_including_agriculture': 0.75
    }
}



#### 4.2 Calculate the area of each land use category

In [ ]:
def calculate_fractional_areas(row):
    """
    Calculate fractional areas for mixed land use types or assign the full area for single use types.
    """
    land_use_type = row['Classify']
    area = row.geometry.area

    # Apply fractional allocation if this is a mixed-use class
    if land_use_type in mixed_land_use_fractions:
        fractions = mixed_land_use_fractions[land_use_type]
        return {category: area * frac for category, frac in fractions.items()}

    # For single-use cases, check which category the land use type belongs to.
    elif land_use_type in forest_flood_equivalent_classes:
        return {'forest_flood_equivalent_classes': area}
    elif land_use_type in afforestable_classes_including_agricultural:
        return {'afforestable_including_agriculture': area}
    else:
        return {'other': area}

In [ ]:
# ----------------------------------------------------------------------------
# Apply the function to each row and "explode" the dictionary so each row has a single key-value pair.
land_use_hydrobasins_intersection_saved['frac_dict'] = land_use_hydrobasins_intersection_saved.apply(calculate_fractional_areas, axis=1)

expanded_rows = []
for idx, row in land_use_hydrobasins_intersection_saved.iterrows():
    for key, value in row['frac_dict'].items():
        new_row = row.copy()
        new_row['LandUseCategory'] = key
        new_row['Area'] = value
        expanded_rows.append(new_row)
        
expanded_gdf = gpd.GeoDataFrame(expanded_rows, crs=land_use_hydrobasins_intersection_saved.crs)
# ----------------------------------------------------------------------------


In [ ]:
# # The "def calculate_fractional_areas" function calculates areas based on land use classifications. 
# # If a land parcel has mixed uses, it allocates the area fractionally among the different uses. 
# # If it’s a single-use land parcel, it assigns the full area to that specific use category.
# # For "mixed_land_use" fractions:
# ## -- The code checks if the current land_use_type is a key in mixed_land_use_fractions. 
# ## -- It retrieves the corresponding fractional allocations.
# ## -- It returns a new dictionary where each sub-type is assigned its respective fractional area (area * frac).

# def calculate_fractional_areas(row):
#     """
#     Calculate the fractional areas for mixed land use types or return the full area for single land use types.
#     """
#     land_use_type = row['Classify']  # Adjust column name if different
#     area = row.geometry.area
    
#     if land_use_type in mixed_land_use_fractions:
#         fractions = mixed_land_use_fractions[land_use_type]
#         return {lu: area * frac for lu, frac in fractions.items()}
#     elif land_use_type in non_afforestable:
#         return {'non_afforestable': area}
#     elif land_use_type in future_afforestable_agricultural_lands:
#         return {'future_afforestable_agricultural_lands': area}
#     elif land_use_type in future_afforestable_non_agricultural_lands:
#         return {'future_afforestable_non_agricultural_lands': area}
#     elif land_use_type in current_forested_lands:
#         return {'current_forested_lands': area}
#     elif land_use_type in tree_plantations:
#         return {'tree_plantations': area}
#     else:
#         return {'other': area}

In [ ]:
# # Step 2: Expand land use areas by fractional components
# expanded_areas = []

# for _, row in land_use_hydrobasins_intersection_saved.iterrows():
#     fractional_areas = calculate_fractional_areas(row)
#     for lu_type, lu_area in fractional_areas.items():
#         expanded_areas.append({
#             'HYBAS_ID': row['HYBAS_ID'],  # Replace with your catchment ID column
#             'LandUseCategory': lu_type,
#             'Area': lu_area,
#             'geometry': row['geometry']  # Keep geometry for plotting
#         })

# # Convert to a GeoDataFrame
# expanded_gdf = gpd.GeoDataFrame(expanded_areas, geometry='geometry')
# display(expanded_gdf)

In [ ]:
# === Update the grouping and summary steps using the new category names ===

# Note: It is assumed that you have applied the calculate_fractional_areas function to your data
# and then "exploded" or restructured the output so that each row of expanded_gdf has:
#   - HYBAS_ID, 
#   - LandUseCategory (the key from the dictionary returned by calculate_fractional_areas),
#   - Area (the corresponding (possibly fractional) area value).

# Calculate total catchment area (if not already computed)
# Here total_catchment_area is assumed to come from earlier processing steps.
# total_catchment_area should be a DataFrame with columns ['HYBAS_ID', 'total_catchment_area']

# Calculate the forest flood equivalent area (i.e. the area that contributes to flood reduction)
forest_flood_equivalent_gdf = expanded_gdf[expanded_gdf['LandUseCategory'] == 'forest_flood_equivalent_classes']
forest_flood_equivalent_area = (
    forest_flood_equivalent_gdf.groupby('HYBAS_ID')['Area'].sum().reset_index()
    .rename(columns={'Area': 'forest_flood_equivalent_area'})
)

# Calculate the afforestable area (including the agricultural component)
afforestable_agri_gdf = expanded_gdf[expanded_gdf['LandUseCategory'] == 'afforestable_including_agriculture']
afforestable_agri_area = (
    afforestable_agri_gdf.groupby('HYBAS_ID')['Area'].sum().reset_index()
    .rename(columns={'Area': 'afforestable_including_agriculture_area'})
)

# Merge the results with the total catchment area DataFrame
final_summary = total_catchment_area.merge(
    forest_flood_equivalent_area, on='HYBAS_ID', how='left'
).merge(
    afforestable_agri_area, on='HYBAS_ID', how='left'
)

# Replace NaN values with 0 if some HYBAS_IDs don't have one of the categories
final_summary['forest_flood_equivalent_area'] = final_summary['forest_flood_equivalent_area'].fillna(0)
final_summary['afforestable_including_agriculture_area'] = final_summary['afforestable_including_agriculture_area'].fillna(0)

# Create a new column for the total future forest area including agriculture
final_summary['total_future_forest_area_including_agri'] = (
    final_summary['forest_flood_equivalent_area'] + final_summary['afforestable_including_agriculture_area']
)

# Calculate percentages relative to the total catchment area
final_summary['forest_flood_equivalent_percentage'] = (
    final_summary['forest_flood_equivalent_area'] / final_summary['total_catchment_area'] * 100
).round(2)
final_summary['afforestable_including_agriculture_percentage'] = (
    final_summary['afforestable_including_agriculture_area'] / final_summary['total_catchment_area'] * 100
).round(2)
final_summary['total_future_forest_including_agri_percentage'] = (
    final_summary['total_future_forest_area_including_agri'] / final_summary['total_catchment_area'] * 100
).round(2)

# Reorder columns for clarity
column_order = [
    'HYBAS_ID', 
    'total_catchment_area',
    'forest_flood_equivalent_area',
    'forest_flood_equivalent_percentage',
    'afforestable_including_agriculture_area',
    'afforestable_including_agriculture_percentage',
    'total_future_forest_area_including_agri',
    'total_future_forest_including_agri_percentage'
]
final_summary = final_summary[column_order]

# Display the final summary DataFrame
display(final_summary)

# Optional: Export the summary to CSV
output_forests = output_path / "catchment_forest_summary_with_percentages.csv"
final_summary.to_csv(output_forests, index=False)
print(f"Data successfully exported to {output_path}")

In [ ]:
# # Calculate current forested area
# current_forested_gdf = expanded_gdf[expanded_gdf['LandUseCategory'] == 'current_forested_lands']
# current_forested_area = (
#     current_forested_gdf.groupby('HYBAS_ID')['Area'].sum().reset_index()
#     .rename(columns={'Area': 'current_forested_area'})
# )

# # Define the categories for future forest area including agriculture
# including_agri_categories = [
#     'current_forested_lands',
#     'future_afforestable_non_agricultural_lands',
#     'future_afforestable_agricultural_lands'
# ]

# # Define the categories for future forest area not including agriculture
# not_including_agri_categories = [
#     'current_forested_lands',
#     'future_afforestable_non_agricultural_lands'
# ]

# # Filter the expanded_gdf for the relevant categories
# including_agri_gdf = expanded_gdf[expanded_gdf['LandUseCategory'].isin(including_agri_categories)]
# not_including_agri_gdf = expanded_gdf[expanded_gdf['LandUseCategory'].isin(not_including_agri_categories)]

# # Group by HYBAS_ID and sum the areas for including agriculture
# future_forest_including_agri = (
#     including_agri_gdf.groupby('HYBAS_ID')['Area'].sum().reset_index()
#     .rename(columns={'Area': 'total_future_forest_area_including_agri'})
# )

# # Group by HYBAS_ID and sum the areas for not including agriculture
# future_forest_not_including_agri = (
#     not_including_agri_gdf.groupby('HYBAS_ID')['Area'].sum().reset_index()
#     .rename(columns={'Area': 'total_future_forest_area_not_including_agri'})
# )

# # Merge results into the final summary DataFrame
# final_summary = total_catchment_area.merge(
#     current_forested_area, on='HYBAS_ID', how='left'
# ).merge(
#     future_forest_including_agri, on='HYBAS_ID', how='left'
# ).merge(
#     future_forest_not_including_agri, on='HYBAS_ID', how='left'
# )

# # Fill NaN values with 0 (if some HYBAS_IDs don't have relevant categories)
# final_summary['current_forested_area'] = final_summary['current_forested_area'].fillna(0)
# final_summary['total_future_forest_area_including_agri'] = final_summary['total_future_forest_area_including_agri'].fillna(0)
# final_summary['total_future_forest_area_not_including_agri'] = final_summary['total_future_forest_area_not_including_agri'].fillna(0)

# # Calculate percentages
# final_summary['current_forested_percentage'] = (
#     final_summary['current_forested_area'] / final_summary['total_catchment_area'] * 100
# ).round(2)
# final_summary['future_forest_including_agri_percentage'] = (
#     final_summary['total_future_forest_area_including_agri'] / final_summary['total_catchment_area'] * 100
# ).round(2)
# final_summary['future_forest_not_including_agri_percentage'] = (
#     final_summary['total_future_forest_area_not_including_agri'] / final_summary['total_catchment_area'] * 100
# ).round(2)

# # Reorder columns to place `current_forested_area` and percentages logically
# column_order = [
#     'HYBAS_ID', 
#     'total_catchment_area', 
#     'current_forested_area', 
#     'current_forested_percentage',
#     'total_future_forest_area_including_agri', 
#     'future_forest_including_agri_percentage',
#     'total_future_forest_area_not_including_agri',
#     'future_forest_not_including_agri_percentage'
# ]
# final_summary = final_summary[column_order]

# # Display the final summary
# display(final_summary)

# print(f"Output path: {output_path}")

# # Optional: Export to CSV
# output_forests = output_path / "catchment_forest_summary_with_percentages.csv"

# #output_shapefile = output_path / "HydroBASINS_Level12_Clipped_Jamaica.shp"

# final_summary.to_csv(output_forests, index=False)
# print(f"Data successfully exported to {output_path}")

In [ ]:
# Step 3: Function to plot a specific land use category (updated for new category names)
def plot_land_use_category(category_name):
    """
    Plot the areas corresponding to a specific land use category.
    """
    # Filter by category
    filtered_data = expanded_gdf[expanded_gdf['LandUseCategory'] == category_name]
    
    if filtered_data.empty:
        print(f"No data found for category '{category_name}'")
        return
    
    # Plotting
    fig, ax = plt.subplots(1, 1, figsize=(12, 12))
    filtered_data.plot(
        column='Area',  # Optionally, color based on area
        cmap='Greens',  # Adjust colormap as needed
        legend=True,
        ax=ax
    )

    # Plot the boundary (assumes jamaica_boundary is a GeoDataFrame with a 'boundary')
    jamaica_boundary.boundary.plot(
        ax=ax, color='black', linewidth=1.5, label='Jamaica Boundary'
    )
    
    # Title and labels
    plt.title(f"Land Use Category: {category_name.replace('_', ' ').title()}", fontsize=16)
    plt.xlabel("Longitude", fontsize=12)
    plt.ylabel("Latitude", fontsize=12)
    
    # Adjust legend position if present
    legend = ax.get_legend()
    if legend:
        legend.set_bbox_to_anchor((1.05, 1), loc='upper left')
    
    plt.tight_layout()
    plt.show()

# Example usage with the updated category name:
plot_land_use_category('afforestable_including_agriculture')

In [ ]:
# # Step 3: Function to plot a specific land use category
# def plot_land_use_category(category_name):
#     """
#     Plot the areas corresponding to a specific land use category.
#     """
#     # Filter by category
#     filtered_data = expanded_gdf[expanded_gdf['LandUseCategory'] == category_name]
    
#     if filtered_data.empty:
#         print(f"No data found for category '{category_name}'")
#         return
    
#     # Plotting
#     fig, ax = plt.subplots(1, 1, figsize=(12, 12))
#     filtered_data.plot(
#         column='Area',  # Optionally, color based on area
#         cmap='Greens',  # Adjust colormap as needed
#         legend=True,
#         ax=ax
#     )

#     jamaica_boundary.boundary.plot(
#         ax=ax, color='black', linewidth=1.5, label='Jamaica Boundary'
#     )
    
#     # Title and labels
#     plt.title(f"Land Use Category: {category_name.replace('_', ' ').title()}", fontsize=16)
#     plt.xlabel("Longitude", fontsize=12)
#     plt.ylabel("Latitude", fontsize=12)
    
#     # Adjust legend
#     legend = ax.get_legend()
#     if legend:
#         legend.set_bbox_to_anchor((1.05, 1), loc='upper left')
    
#     plt.tight_layout()
#     plt.show()

# # Example usage:
# plot_land_use_category('future_afforestable_agricultural_lands')

In [ ]:
# Step 1: Aggregate Area by Catchment and Land Use Category
category_area = expanded_gdf.groupby(['HYBAS_ID', 'LandUseCategory'])['Area'].sum().reset_index()

# Step 2: Calculate Total Area per Catchment from expanded_gdf
total_area = expanded_gdf.groupby('HYBAS_ID')['Area'].sum().reset_index().rename(columns={'Area': 'TotalArea'})

# Step 3: Merge Aggregated Data with Total Area
category_percentage = pd.merge(category_area, total_area, on='HYBAS_ID')

# Step 4: Compute Percentage per Land Use Category
category_percentage['Percentage'] = (category_percentage['Area'] / category_percentage['TotalArea']) * 100

# (Optional) Step 5: Pivot Data for Easier Interpretation
percentage_pivot = category_percentage.pivot(index='HYBAS_ID', columns='LandUseCategory', values='Percentage').fillna(0).reset_index()

# Display the percentage DataFrame
display("Percentage of Each Land Use Category within Each Catchment:")
display(percentage_pivot.head())

# (Optional) Step 6: Merge Percentages Back with Geometry for Spatial Analysis or Mapping
# If you need to visualize or further analyze, you might want to have the geometry associated with each HYBAS_ID.
# Assuming you have a GeoDataFrame 'hydrobasins_gdf' with 'HYBAS_ID' and 'geometry':
#
# hydrobasins_gdf = gpd.read_file('path_to_hydrobasins.shp')
# percentage_with_geometry = pd.merge(percentage_pivot, hydrobasins_gdf[['HYBAS_ID', 'geometry']], on='HYBAS_ID')
# percentage_gdf = gpd.GeoDataFrame(percentage_with_geometry, geometry='geometry')
#
# Now you can plot or analyze 'percentage_gdf' as needed.

In [ ]:
# # Assume 'expanded_gdf' is your GeoDataFrame from the previous steps
# # It should have columns: 'HYBAS_ID', 'LandUseCategory', 'Area', 'geometry'

# # Step 1: Aggregate Area by Catchment and Land Use Category
# category_area = expanded_gdf.groupby(['HYBAS_ID', 'LandUseCategory'])['Area'].sum().reset_index()

# # Step 2: Calculate Total Area per Catchment
# total_area = expanded_gdf.groupby('HYBAS_ID')['Area'].sum().reset_index().rename(columns={'Area': 'TotalArea'})

# # Step 3: Merge Aggregated Data with Total Area
# category_percentage = pd.merge(category_area, total_area, on='HYBAS_ID')

# # Step 4: Compute Percentage per Land Use Category
# category_percentage['Percentage'] = (category_percentage['Area'] / category_percentage['TotalArea']) * 100

# # (Optional) Step 5: Pivot Data for Easier Interpretation
# percentage_pivot = category_percentage.pivot(index='HYBAS_ID', columns='LandUseCategory', values='Percentage').fillna(0).reset_index()

# # Display the percentage DataFrame
# print("Percentage of Each Land Use Category within Each Catchment:")
# print(percentage_pivot.head())

# # (Optional) Step 6: Merge Percentages Back with Geometry for Spatial Analysis or Mapping
# # If you need to visualize or further analyze, you might want to have the geometry associated with each HYBAS_ID
# # Assuming you have a GeoDataFrame 'hydrobasins_gdf' with 'HYBAS_ID' and 'geometry'

# # Example:
# # hydrobasins_gdf = gpd.read_file('path_to_hydrobasins.shp')
# # percentage_with_geometry = pd.merge(percentage_pivot, hydrobasins_gdf[['HYBAS_ID', 'geometry']], on='HYBAS_ID')
# # percentage_gdf = gpd.GeoDataFrame(percentage_with_geometry, geometry='geometry')

# # # Now you can plot or analyze 'percentage_gdf' as needed

In [ ]:
# Display the first few rows to verify
display(percentage_pivot.head())

In [ ]:
display("Aggregated summary by HYBAS_ID:")
display(final_summary.head())

In [ ]:
# Define the output path and export percentage pivot DataFrame
output_hydrobasins_percentage_afforestable_csv_path = output_path / "afforestable_percentages_by_catchment.csv"
percentage_pivot.to_csv(output_hydrobasins_percentage_afforestable_csv_path, index=False)

In [ ]:
# 1. Pivot the area data by HYBAS_ID and LandUseCategory
area_pivot = category_percentage.pivot(
    index='HYBAS_ID',
    columns='LandUseCategory',
    values='Area'
).fillna(0).reset_index()

# 2. Merge the area data into percentage_pivot
percentage_pivot = pd.merge(percentage_pivot, area_pivot, on='HYBAS_ID', suffixes=('_pct', '_area'))

# 3. Add the total area for each hydrobasin.
# This assumes that the area columns from the pivot start at a fixed position; adjust if needed.
percentage_pivot['TotalArea'] = percentage_pivot.iloc[:, len(percentage_pivot.columns) - len(area_pivot.columns) + 1:].sum(axis=1)

# Define the export path for percentages including catchment area and export the DataFrame
output_hydrobasins_percentage_afforestable_with_catchment_area_csv_path = output_path / "afforestable_percentages_by_catchment_including_area.csv"
percentage_pivot.to_csv(output_hydrobasins_percentage_afforestable_with_catchment_area_csv_path, index=False)

# Add a new column for TotalArea in km² by dividing by 1,000,000
percentage_pivot['TotalArea_km2'] = percentage_pivot['TotalArea'] / 1_000_000

# Display the updated DataFrame to check the new column
display(percentage_pivot.head())

In [ ]:
# Export the final DataFrame with total afforestable area by catchment
total_afforestable_area_by_catchment_csv_path = output_path / "total_afforestable_area_by_catchment.csv"
percentage_pivot.to_csv(total_afforestable_area_by_catchment_csv_path, index=False)